# Frozen truth + MMR: impact of weights and values (replayable)

`daisy/TRUTH-001.json` is frozen (its id hashes the toy KG root, dataset root, protocol and every subject; editing it breaks the id). `daisy/impact_mmr.jsonl` is an append-only Merkle Mountain Range: leaf 0 = truth, then one event per (actor weights × values) and a consensus event per values. Every event has an inclusion proof against any later root.

New weights (Kaggle DAISY runs, new models) are **appended**: `python -m baebugs.impact append --actor-name NAME --weights SHA --results lomo.jsonl` or `replay` after adding result files.

In [1]:
import os, sys, json, csv
from pathlib import Path
if Path.cwd().name == 'notebooks': os.chdir('..')
sys.path.insert(0, os.getcwd())
from baebugs import impact
from baebugs.mmr import MMRLog, MMR
t = impact.load_truth(); print('truth', t['id'][:16], t['sources'])
log = MMRLog(impact.LOG); print('MMR size', len(log.records), 'root', log.mmr.root[:16], 'verify', log.verify())

truth 5d190ae6be4b3b6b {'toy_kg_collection_root': 'ddbf08d2bf382a2dcf6fcb85afa259122e5ab1a20bb90707df68c5d4104371cc', 'toy_index_sha256': 'b09b3f48e63447422789d4236c96c692063a631c0ed78e04db34d77490606ef8', 'daisy_datasets_root': '5a0d1661538f2a9c1338a274f90477cade19ed034bb09b607ee08ec82004bd86', 'daisy_protocol_sha256': '9449fd7c0724f0e3813be357800ac7e286d64b75926b5c8ce205f0a59ae8cb63', 'code_commit': '5f88331851bb459cf0d6df77cc35f58a446c313b'}
MMR size 53 root 953ea7745db83d38 verify (True, None)


## Impact table (pass = claims reproduced within the values' tolerance)

In [2]:
import pandas as pd
df = pd.read_csv('daisy/impact_events.csv')
ev = df[df.type=='evaluation'].pivot_table(index='actor', columns='values', values='pass')
ev[['exact','tol_0.5mm','tol_2mm','plan_only']]

values,exact,tol_0.5mm,tol_2mm,plan_only
actor,,,,
classical@10px/mm,17.0,76.0,76.0,76.0
classical@20px/mm,32.0,79.0,79.0,79.0
classical@5px/mm,12.0,60.0,60.0,60.0
lfm2.5-1.2b-instruct_q4_k_m,0.0,3.0,5.0,25.0
logic-twin,86.0,87.0,87.0,87.0
qwen3_1.7b,0.0,1.0,3.0,10.0
yolo@10px/mm,0.0,2.0,2.0,4.0
yolo@20px/mm,2.0,21.0,21.0,22.0
yolo@5px/mm,0.0,0.0,0.0,5.0


In [3]:
df[df.type=='consensus'][['values','status_counts','mmr_root']]

,values,status_counts,mmr_root
13,exact,"{""DISPUTED"": 86, ""FAILED"": 5}",f891b67c59d8601c
26,tol_0.5mm,"{""DISPUTED"": 72, ""SUPPORTED"": 15, ""FAILED"": 4}",4d03884a7e3b856e
39,tol_2mm,"{""DISPUTED"": 72, ""SUPPORTED"": 15, ""FAILED"": 4}",f032451bab91880d
52,plan_only,"{""DISPUTED"": 69, ""SUPPORTED"": 18, ""FAILED"": 4}",953ea7745db83d38


## Proof: an early event is still in today's history

In [4]:
p = log.mmr.proof(1)
print('event 1 =', log.records[1]['record']['actor']['name'], log.records[1]['record']['values'], '| proof valid vs current root:', MMR.verify(p, log.mmr.root))

event 1 = logic-twin exact | proof valid vs current root: True
